# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 236.97it/s]


2026-04-23 11:02:30.698 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:907 - Data batch-empirical estimation of propensity score.


2026-04-23 11:02:30.707 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:956 - Data prediction of expected reward based on gbm model.


2026-04-23 11:02:32.132 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1073 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-04-23 11:03:03.951 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-23 11:03:04.346 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-23 11:03:04.743 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.496381,0.463521,0.529810,0.016898,b-ipw,reward_0
1,0.499360,0.498772,0.499976,0.000309,dm,reward_0
2,0.495908,0.464419,0.528412,0.016421,dr,reward_0
3,0.499360,0.498773,0.499976,0.000304,dros-opt,reward_0
4,0.495908,0.463779,0.527076,0.016375,dros-pess,reward_0
5,0.495778,0.462925,0.528979,0.017078,ipw,reward_0
6,0.495688,0.462428,0.529678,0.017093,rep,reward_0
7,0.495907,0.463471,0.527531,0.016414,sndr,reward_0
8,0.495895,0.462683,0.529252,0.017023,snips,reward_0
9,0.495908,0.463455,0.527666,0.016504,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 270.39it/s]


2026-04-23 11:03:05.304 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1305 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<09:03,  1.84it/s]

...

2026-04-23 11:03:52.723 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-23 11:03:53.122 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-23 11:03:53.520 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.497586,0.467571,0.529523,0.015860,b-ipw,reward_0
1,0.499236,0.498561,0.499911,0.000345,dm,reward_0
2,0.500402,0.468489,0.531162,0.016039,dr,reward_0
3,0.499236,0.498553,0.499891,0.000341,dros-opt,reward_0
4,0.500402,0.470236,0.532420,0.015896,dros-pess,reward_0
5,0.500782,0.468926,0.532327,0.016145,ipw,reward_0
6,0.500249,0.468212,0.532383,0.016192,rep,reward_0
7,0.500401,0.470177,0.532015,0.015958,sndr,reward_0
8,0.500371,0.469748,0.532628,0.016020,snips,reward_0
9,0.500402,0.469841,0.531999,0.015842,sg-dr,reward_0
